## RAG with Vector Search  

Now that we can run a search with vectors, let's implement it in a RAG system.

In [15]:
# Boilerplate we previously used to build the RAG system
"""
def rag(question):
    search_results = search(question) # Retrieval
    user_prompt = build_prompt(question, search_results) # Augumentation
    return llm(user_prompt) # Generation
"""

'\ndef rag(question):\n    search_results = search(question) # Retrieval\n    user_prompt = build_prompt(question, search_results) # Augumentation\n    return llm(user_prompt) # Generation\n'

In [16]:
# Create OpenAI client
from dotenv import load_dotenv
from openai import OpenAI

# loads the environment variables from the .env file
load_dotenv()
# OpenAI() with no arguments automatically looks for OPENAI_API_KEY in the environment and uses it
openai_client = OpenAI()

In [17]:
from ingest import load_faq_data, build_index

# Retrieve the knowledge base (FAQs) -> list[dicts]
text_documents = load_faq_data()

# Build the minsearch index -> minsearch.Index
index = build_index(text_documents)

In [18]:
# Create the RAG assistant object from our class -> rag_helper.RAGBase
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [19]:
# Test the RAG system by asking a question
# Retrieval is still text-based: ingest.build_index() creates a minsearch.Index
# RAGBase.search() queries it before sending context to the LLM 
user_query = "I just found out about the program, can I still sign up?"
assistant.rag_answer(user_query)

'Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still open.'

### Preparing the dataset to implement the Vector Search

`text_documents` is a list[dicts].  
The embedder only accepts plain text, so we turn each FAQ from `text_documents` into one string (`"question: answer"`) before calling `encode()`.  

We keep the original dicts in `text_documents` for everything else (search results, filtering by course).

In [20]:
faq_texts = [f"{dict_doc['FAQ']} Answer: {dict_doc['answer']}" for dict_doc in text_documents]

In [21]:
# Import the embedder model and initialize it
# The first time, this import will download locaally the model from Hugging Face
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [22]:
# tqdm provides a visual progress bar for aesthetic purposes
from tqdm.auto import tqdm

In [23]:
batch_size = 50
# Initialize an empty list to store the vectors -> list[vectors]
documents_vectors = []

for i in tqdm(range(0, len(text_documents), batch_size)):
    # Get the current batch of texts
    documents_batch = faq_texts[i:i+batch_size]
    # Embed the current batch
    current_batch_vectors = embedder.encode(documents_batch)
    # Save current batch's vectors by adding them to the list
    documents_vectors.extend(current_batch_vectors)

print(f"Done! We have a matrix with {len(documents_vectors)} vector rows.")

  0%|          | 0/28 [00:00<?, ?it/s]

Done! We have a matrix with 1353 vector rows.


>**NOTE** `documents_vectors` looks matrix-like, but it isn’t one yet: it’s a Python list of vectors. NumPy turns that list of vectors into a single matrix so search is fast.

In [24]:
# Import numpy to convert the list of vectors into a matrix to enable fast vector operations
import numpy as np

# Convert the list of vectors into a matrix
documents_matrix = np.array(documents_vectors)

In [25]:
from rag_helper import RAGVector
from minsearch import VectorSearch

In [26]:
# Create an empty VectorSearch object (index for vector search)
vector_index = VectorSearch(keyword_fields=["course"])

# Create the Index that matched the vectorized documents to the actual documents
vector_index.fit(documents_matrix, text_documents)

In [27]:
# Initialize the RAGVector object to run the RAG
vector_assistant = RAGVector(
    embedder=embedder,
    index=vector_index,
    llm_client=openai_client,
)


In [28]:
vector_assistant.rag_answer("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

# Implementing Persistent Memory with `sqlitesearch`

## Limits of `minsearch`

1. **It rebuilds the index on every startup.**
2. **It keeps everything in volatile memory (RAM)**

 With **text search** **indexing** was fast because we didn't embed anything. With **vector search**, **indexing** runs a neural network over every document, so running it at every startup can take time based on the dataset size.  

 Embedding the whole dataset to build the index takes about a minute (in our case). We don't want a user waiting that long when the app starts up. We pay that cost once during ingestion, and the query side starts up instantly.  
 
 Keeping everything in memory is fine here, but a larger dataset would need too much space.

3. **It searches by brute force.**

For every query we compare the query vector against every single document. With 1,000 documents this is fine, probably even faster than anything smarter. But as the dataset grows past 10,000 or so, it gets slow, and we'll want an approximate method instead.

> What we've done so far is ***exact nearest neighbor (NN) search***. We score the query against every document and pick the top ones. It always finds the true top matches, but it is resource intensive for that by touching everything.



**Approximate nearest neighbor (ANN)** search takes a shortcut. Instead of comparing against everything, it first narrows down to a region of likely matches. Then it scores only within that region. It may miss the absolute best match, but the results are still good and it's much faster.

> **NN (exact)**:    compare query against ALL documents →  top 5
**ANN (approximate)**:  narrow down to a region → compare within region → top 5

## Implementation of `sqlitesearch`

`sqlitesearch` supports three ANN modes:

- ***lsh*** (default): up to 100K vectors, random hyperplane projections
- ***ivf***: 10K-500K vectors, K-means clustering
- ***hnsw***: 10K-1M+ vectors, proximity graph (highest recall)

For our small dataset, *lsh* is fine.  
All modes use two-phase search: approximate candidate retrieval, then exact cosine similarity reranking.

In [29]:
import sqlitesearch

NOTE: We are essentially using a general-purpose database (SQLite) to store specialized data structures (the index) for a specific purpose (vector search).

In [ ]:
# Initialize an empty vector database
vs_index = sqlitesearch.VectorSearchIndex(
    mode="lsh",
    keyword_fields=["course"],
    db_path="zoomcamps.db",
)

The `fit` method takes the raw vectors and text documents and writes them into the SQLite file (.db) specified when we initialized the index.

**Note on structure creation**: `fit()` doesn't just "dump" the data; it builds the internal data structures (the "index" itself) within that SQLite container so that future search queries can efficiently navigate the data.

In [31]:
# Let's populate the index with the vectors and documents
vs_index.fit(documents_matrix, text_documents)

In [33]:
# User asks a question
query = "I just discovered the course. Can I still join it?"
# Embed the query
query_vector = embedder.encode(query)
# Search the index to find the most similar vectors and returns the mapped documents
results = vs_index.search(query_vector, num_results=5)

In [35]:
# Checkl out the results they include answer from more than one course
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf',
  'FAQ': 'I just discovered the course. Can I still join?'},
 {'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'answer': "No, you can only get a certificate if you finish the course with a “live” cohort. We don't award certificates for the self-paced mode. The reason is you need to peer-review capstone(s) after submitting a project. You can only peer-review projects at the time the course is running.",
  'doc_id': '900f60fd25',
  'FAQ': 'Certificate - Can I follow the course in a self-paced mode and get a certificate?'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'answer': "No, you can only get a certificate if you finish the course with a “live” cohort. We don't award cert

### Adding filtering by course

In [37]:
results = vs_index.search(query_vector, num_results=5, filter_dict={"course": "llm-zoomcamp"})

In [39]:
# Now the resulrs will only include answers from the llm-zoomcamp course
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf',
  'FAQ': 'I just discovered the course. Can I still join?'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e',
  'FAQ': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?'},
 {'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to start

>Calling `.close()` acts as a "save" command, ensuring that all pending changes are flushed from memory to the disk. If we exit the program without closing it, we can risk corrupting the database file or losing the most recent updates.

In [40]:
# When done close the index
vs_index.close()

### Reopening the index
In a new Python session, you can reopen the index without re-computing embeddings as they are saved persistently in the hard drive (or whatever database server).

#### What is not persistent when we start a new Python session
The embeeding model's object (which performs the complex math to convert text to vectors) and the *index* object (which manages the connection to our db file) no longer exist in the computer's memory.  

We still load the embedding model to encode the query, but **we don't re-embed all the documents**. No `fit` call needed, because **the index is already built and waiting on disk**.

In [43]:
# What we would when we start a new Python session
"""
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)
"""

'\nfrom sentence_transformers import SentenceTransformer\nfrom sqlitesearch import VectorSearchIndex\n\nmodel = SentenceTransformer("all-MiniLM-L6-v2")\n\nvs_index = VectorSearchIndex(\n    keyword_fields=["course"],\n    mode="ivf",\n    db_path="faq_vectors2.db"\n)\n'

In [45]:
# Then when a new user asks a question we can just do the search:
"""
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)
"""

'\nquery_vector = model.encode("How do I run Kafka?")\nresults = vs_index.search(query_vector, num_results=5)\n'

## Implementing `sqlitesearch` vector search in the RAG pipeline

See `noteboook_VS_RAG.ipynb`